Extraction de MongoDB et chargement dans Polars

In [56]:
# Import des dépendances
from pymongo import MongoClient
import polars as pl

# Connexion MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["locparis"]
collection = db["listings"]

# Projection : on ne récupère que les champs utiles
projection = {
    "_id": 0,
    "host_is_superhost": 1,
    "room_type": 1,
    "neighbourhood_cleansed": 1,
    "number_of_reviews": 1,
    "availability_365": 1,
}
docs = list(collection.find({}, projection))

# Chargement dans Polars
df = pl.DataFrame(docs)

# Nettoyage / typage
df = df.with_columns([
    pl.col("host_is_superhost").cast(pl.Utf8, strict=False),
    pl.col("room_type").cast(pl.Utf8, strict=False),
    pl.col("neighbourhood_cleansed").cast(pl.Utf8, strict=False),
    pl.col("number_of_reviews").cast(pl.Int64, strict=False),
    pl.col("availability_365").cast(pl.Int64, strict=False),
])

print(df.head())
print(df.schema)

shape: (5, 5)
┌───────────────────┬────────────────────────┬─────────────────┬──────────────────┬───────────────────┐
│ host_is_superhost ┆ neighbourhood_cleansed ┆ room_type       ┆ availability_365 ┆ number_of_reviews │
│ ---               ┆ ---                    ┆ ---             ┆ ---              ┆ ---               │
│ str               ┆ str                    ┆ str             ┆ i64              ┆ i64               │
╞═══════════════════╪════════════════════════╪═════════════════╪══════════════════╪═══════════════════╡
│ f                 ┆ Ménilmontant           ┆ Entire home/apt ┆ 0                ┆ 206               │
│ f                 ┆ Observatoire           ┆ Entire home/apt ┆ 349              ┆ 4                 │
│ t                 ┆ Hôtel-de-Ville         ┆ Entire home/apt ┆ 277              ┆ 354               │
│ f                 ┆ Hôtel-de-Ville         ┆ Entire home/apt ┆ 154              ┆ 403               │
│ f                 ┆ Opéra                  ┆ Ent

Taux de réservation moyen par mois par type de logement

In [57]:
df_reservation = (df
    .filter(
        (pl.col("availability_365") >= 0) &
        (pl.col("availability_365") <= 365) &
        pl.col("availability_365").is_not_null()
    )
    .group_by("room_type")
    .agg(
        jours_reserves_moyens_par_mois =
        ((365 - pl.col("availability_365")) / 12)
        .mean()
        .round(2)
    )
    .sort("jours_reserves_moyens_par_mois", descending=True)
)
df_reservation

room_type,jours_reserves_moyens_par_mois
str,f64
"""Private room""",20.93
"""Entire home/apt""",19.74
"""Shared room""",19.26
"""Hotel room""",15.93


La médiane des nombre d’avis pour tous les logements

In [58]:
df_mediane = (df
    .filter(
        (pl.col("number_of_reviews") >= 0) &
        pl.col("number_of_reviews").is_not_null()
    )
    .select(
        pl.col("number_of_reviews")
        .median()
        .alias("mediane_nombre_avis")
    )
)
df_mediane

mediane_nombre_avis
f64
3.0


La médiane des nombre d’avis par catégorie d’hôte

In [59]:
df.group_by("host_is_superhost").len()

host_is_superhost,len
str,u32
"""f""",81035
"""t""",14776
"""""",74


In [60]:
df_mediane_superhost = (df
    .filter(
        pl.col("host_is_superhost").is_in(["t", "f"]) &
        (pl.col("number_of_reviews") >= 0) &
        pl.col("number_of_reviews").is_not_null()
    )
    .with_columns(
        pl.when(pl.col("host_is_superhost") == "t")
        .then(pl.lit("Super hôte"))
        .otherwise(pl.lit("Non super hôte"))
        .alias("categorie_hote")
    )
    .group_by("categorie_hote")
    .agg(
        mediane_nombre_avis = pl.col("number_of_reviews")
        .median()
    )
    .sort("mediane_nombre_avis", descending=True)
)
df_mediane_superhost

categorie_hote,mediane_nombre_avis
str,f64
"""Super hôte""",24.0
"""Non super hôte""",2.0


La densité de logements par quartier de Paris

In [61]:
df_densite_quartier = (df
    .filter(
        pl.col("neighbourhood_cleansed").is_not_null() &
        (pl.col("neighbourhood_cleansed") != "")
    )    
    .group_by("neighbourhood_cleansed")
    .agg(nombre_logements = pl.len())
    .with_columns(
        part_logements_pct = (
            pl.col("nombre_logements") * 100 
            / pl.col("nombre_logements").sum()
        )
        .round(2)
    )
    .sort("nombre_logements", descending=True)
)
df_densite_quartier

neighbourhood_cleansed,nombre_logements,part_logements_pct
str,u32,f64
"""Buttes-Montmartre""",10555,11.01
"""Popincourt""",8430,8.79
"""Vaugirard""",7802,8.14
"""Batignolles-Monceau""",6857,7.15
"""Entrepôt""",6558,6.84
…,…,…
"""Élysée""",2898,3.02
"""Hôtel-de-Ville""",2821,2.94
"""Palais-Bourbon""",2740,2.86


Les quartiers avec le plus fort taux de réservation par mois

In [62]:
df_reservation_quartier = (df
    .filter(
        pl.col("neighbourhood_cleansed").is_not_null() &
        (pl.col("neighbourhood_cleansed") != "") &
        pl.col("availability_365").is_not_null() &
        (pl.col("availability_365") >= 0) &
        (pl.col("availability_365") <= 365)
    )
    .group_by("neighbourhood_cleansed")
    .agg(
        taux_reservation_moyen_mensuel = (
            ((365 - pl.col("availability_365")) / 12)
            .mean()
            .round(2)
        ),
        nombre_logements = pl.len()
    )
    .sort("taux_reservation_moyen_mensuel", descending=True)
)
df_reservation_quartier.head(10)

neighbourhood_cleansed,taux_reservation_moyen_mensuel,nombre_logements
str,f64,u32
"""Ménilmontant""",21.62,5271
"""Buttes-Chaumont""",21.21,5465
"""Buttes-Montmartre""",21.06,10555
"""Entrepôt""",20.99,6558
"""Popincourt""",20.95,8430
"""Gobelins""",20.76,3304
"""Reuilly""",20.61,4003
"""Panthéon""",20.18,3010
"""Vaugirard""",20.05,7802
